In [1]:
# Config: paths and output
import os
DATA_DIR = "datathon_dataset"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
# Load datasets via config paths (with guard if config not run)
try:
    DATA_DIR
except NameError:
    import os
    DATA_DIR = "datathon_dataset"
    TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
    TEST_PATH = os.path.join(DATA_DIR, "test.csv")
    SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
    OUTPUT_DIR = "outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

import pandas as pd
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SAMPLE_SUB_PATH)


In [ ]:
# Guard: stub matplotlib to avoid NumPy 1.x ABI import crash
import sys, types
if 'matplotlib' not in sys.modules:
    sys.modules['matplotlib'] = types.ModuleType('matplotlib')


In [3]:
import pandas as pd
import numpy as np

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
submission  = pd.read_csv(SAMPLE_SUB_PATH)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141219 entries, 0 to 141218
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     141219 non-null  object 
 1   event_type     141219 non-null  object 
 2   product_id     141219 non-null  object 
 3   category_id    141219 non-null  object 
 4   user_id        141219 non-null  object 
 5   user_session   141219 non-null  object 
 6   session_value  141219 non-null  float64
dtypes: float64(1), object(6)
memory usage: 7.5+ MB


In [6]:
unique_counts = train.nunique()

train["event_time"] = pd.to_datetime(train["event_time"])

min_date = train["event_time"].min()
max_date = train["event_time"].max()
unique_days = train["event_time"].dt.date.nunique()

print("Unique değer sayıları:\n", unique_counts)
print("\nEn küçük tarih:", min_date)
print("En büyük tarih:", max_date)
print("Kaç farklı gün:", unique_days)

test_unique_counts = test.nunique()

test["event_time"] = pd.to_datetime(test["event_time"])

test_min_date = test["event_time"].min()
test_max_date = test["event_time"].max()
test_unique_days = test["event_time"].dt.date.nunique()

print("Unique değer sayıları:\n", test_unique_counts)
print("\nEn küçük tarih:", test_min_date)
print("En büyük tarih:", test_max_date)
print("Kaç farklı gün:", test_unique_days)

Unique değer sayıları:
 event_time       128357
event_type            4
product_id        26470
category_id         448
user_id           51821
user_session      70736
session_value     12719
dtype: int64

En küçük tarih: 2025-06-01 00:00:24+00:00
En büyük tarih: 2025-06-21 23:59:52+00:00
Kaç farklı gün: 21
Unique değer sayıları:
 event_time      56682
event_type          4
product_id      17450
category_id       433
user_id         22665
user_session    30789
dtype: int64

En küçük tarih: 2025-06-22 00:01:00+00:00
En büyük tarih: 2025-06-30 23:59:47+00:00
Kaç farklı gün: 9


In [7]:
train["event_type"].unique()

array(['ADD_CART', 'VIEW', 'REMOVE_CART', 'BUY'], dtype=object)

In [8]:
import pandas as pd
import numpy as np

# Zaman tipine çevir
train['event_time'] = pd.to_datetime(train['event_time'])
test['event_time'] = pd.to_datetime(test['event_time'])
train['event_date'] = train['event_time'].dt.date
test['event_date'] = test['event_time'].dt.date

print("Zaman döngüsüne dayalı özellikler ekleniyor...")

train['day_of_week'] = train['event_time'].dt.dayofweek
test['day_of_week'] = test['event_time'].dt.dayofweek
train['hour'] = train['event_time'].dt.hour
test['hour'] = test['event_time'].dt.hour
print("Zaman tabanlı mevcut özellikler başarıyla eklendi.")

all_event_types = ['ADD_CART', 'VIEW', 'REMOVE_CART', 'BUY']

# -----------------------------
# Kimlik bazlı toplam sayım
# -----------------------------
def calculate_total_counts(df, id_col, all_event_types):
    df_counts = (
        df.pivot_table(
            index=id_col, 
            columns='event_type', 
            aggfunc='size',  
            fill_value=0
        )
        .reindex(columns=all_event_types, fill_value=0)
        .reset_index()
    )
    df_counts.columns = [id_col] + [f"{id_col.split('_')[0]}_{et.lower()}_count" for et in all_event_types]
    return df_counts

# -----------------------------
# Oturum ve zaman tabanlı özellikler
# -----------------------------
def calculate_session_features(df):
    df.sort_values(by=['user_session', 'event_time'], inplace=True)
    session_start_time = df.groupby('user_session')['event_time'].transform('min')
    session_end_time = df.groupby('user_session')['event_time'].transform('max')
    
    df['time_since_session_start'] = (df['event_time'] - session_start_time).dt.total_seconds()
    df['session_event_count'] = df.groupby('user_session')['event_time'].transform('count')
    df['session_product_count'] = df.groupby('user_session')['product_id'].transform('nunique')
    df['session_category_count'] = df.groupby('user_session')['category_id'].transform('nunique')
    df['time_to_next_event'] = df.groupby('user_session')['event_time'].diff(periods=-1).dt.total_seconds().abs()
    df['is_last_event_of_session'] = (df['event_time'] == session_end_time).astype(int)
    df['session_duration'] = (session_end_time - session_start_time).dt.total_seconds()
    
    first_add_cart_time = df[df['event_type'] == 'ADD_CART'].groupby('user_session')['event_time'].transform('min')
    first_buy_time = df[df['event_type'] == 'BUY'].groupby('user_session')['event_time'].transform('min')
    df['time_to_first_add_cart'] = (first_add_cart_time - session_start_time).dt.total_seconds()
    df['time_to_first_buy'] = (first_buy_time - session_start_time).dt.total_seconds()
    
    # Eklenen özellikler
    df['session_unique_days'] = df.groupby('user_session')['event_time'].transform(lambda x: x.dt.date.nunique())
    df['session_day_span'] = df.groupby('user_session')['event_time'].transform(lambda x: (x.max().date() - x.min().date()).days)
    df['session_duration_minutes'] = df['session_duration'] / 60
    df['session_daily_avg_events'] = df['session_event_count'] / df['session_unique_days'].replace(0, 1)
    
    # Günün saat dilimleri
    df['morning_event'] = df['event_time'].dt.hour.between(6, 11).astype(int)
    df['afternoon_event'] = df['event_time'].dt.hour.between(12, 17).astype(int)
    df['evening_event'] = df['event_time'].dt.hour.between(18, 23).astype(int)
    df['night_event'] = df['event_time'].dt.hour.between(0, 5).astype(int)
    
    inter_event_diff = df.groupby('user_session')['event_time'].diff().dt.total_seconds()
    df['mean_inter_event_sec'] = inter_event_diff.groupby(df['user_session']).transform('mean').fillna(0)
    df['std_inter_event_sec'] = inter_event_diff.groupby(df['user_session']).transform('std').fillna(0)
    
    return df

# -----------------------------
# Günlük ve oturum-günlük oran
# -----------------------------
def add_daily_and_session_to_daily_features(df):
    print("Günlük ve oturum-günlük oran özellikleri hesaplanıyor...")
    daily_stats = df.groupby('event_date').agg(
        daily_event_count=('event_time', 'count'),
        daily_unique_users=('user_id', 'nunique'),
        daily_unique_products=('product_id', 'nunique')
    ).reset_index()

    daily_event_counts = df.groupby(['event_date', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0)
    daily_event_counts.columns = [f"daily_{col.lower()}_count" for col in daily_event_counts.columns]

    df = pd.merge(df, daily_stats, on='event_date', how='left')
    df = pd.merge(df, daily_event_counts.reset_index(), on='event_date', how='left')
    
    for etype in all_event_types:
        df[f'daily_{etype.lower()}_rate'] = df[f'daily_{etype.lower()}_count'] / df['daily_event_count']

    session_event_counts = df.groupby(['user_session', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0).reset_index()
    df = pd.merge(df, session_event_counts, on='user_session', how='left')
    for etype in all_event_types:
        df[f'session_to_daily_{etype.lower()}_ratio'] = df[etype] / df[f'daily_{etype.lower()}_count']
        
    df.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    return df

# -----------------------------
# Kapsamlı user features
# -----------------------------
def calculate_comprehensive_user_features(df):
    print("Kapsamlı user-id bazlı özellikler hesaplanıyor...")
    user_agg_df = df.groupby('user_id').agg(
        user_total_events=('event_type', 'count'),
        user_unique_sessions=('user_session', 'nunique'),
        user_unique_products=('product_id', 'nunique'),
        user_unique_categories=('category_id', 'nunique'),
        user_min_event_time=('event_time', 'min'),
        user_max_event_time=('event_time', 'max'),
        avg_time_between_events=('event_time', lambda x: x.diff().mean().total_seconds()),
        std_time_between_events=('event_time', lambda x: x.diff().std().total_seconds())
    ).reset_index()

    user_agg_df['user_total_duration'] = (user_agg_df['user_max_event_time'] - user_agg_df['user_min_event_time']).dt.total_seconds()

    user_unique_days = df.groupby('user_id')['event_time'].apply(lambda x: x.dt.date.nunique()).reset_index(name='user_unique_days')
    user_avg_events_per_day = df.groupby('user_id')['event_time'].apply(lambda x: x.count()/x.dt.date.nunique()).reset_index(name='user_avg_events_per_day')
    user_most_active_day = df.groupby('user_id')['event_time'].apply(lambda x: x.dt.dayofweek.value_counts().idxmax()).reset_index(name='user_most_active_day')
    user_days_range = df.groupby('user_id')['event_time'].apply(lambda x: (x.max().date() - x.min().date()).days).reset_index(name='user_days_range')

    user_agg_df = pd.merge(user_agg_df, user_unique_days, on='user_id', how='left')
    user_agg_df = pd.merge(user_agg_df, user_avg_events_per_day, on='user_id', how='left')
    user_agg_df = pd.merge(user_agg_df, user_most_active_day, on='user_id', how='left')
    user_agg_df = pd.merge(user_agg_df, user_days_range, on='user_id', how='left')

    user_event_counts = df.groupby(['user_id', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0).reset_index()
    user_event_counts.columns = ['user_id'] + [f"user_{col.lower()}_count" for col in all_event_types]
    user_agg_df = pd.merge(user_agg_df, user_event_counts, on='user_id', how='left')

    for etype in all_event_types:
        user_agg_df[f'user_{etype.lower()}_rate'] = user_agg_df[f'user_{etype.lower()}_count'] / user_agg_df['user_total_events']

    user_agg_df['user_view_to_add_cart_ratio'] = user_agg_df['user_view_count'] / user_agg_df['user_add_cart_count']
    user_agg_df['user_add_cart_to_buy_ratio'] = user_agg_df['user_add_cart_count'] / user_agg_df['user_buy_count']
    user_agg_df['user_buy_to_total_ratio'] = user_agg_df['user_buy_count'] / user_agg_df['user_total_events']
    user_agg_df['user_cart_abandon_ratio'] = (user_agg_df['user_add_cart_count'] - user_agg_df['user_buy_count']) / user_agg_df['user_add_cart_count']

    user_agg_df.rename(columns={'avg_time_between_events': 'user_avg_time_between_events',
                                'std_time_between_events': 'user_std_time_between_events'}, inplace=True)

    user_agg_df.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    user_agg_df = pd.merge(user_agg_df, df.groupby('user_id')['session_duration'].mean().reset_index(name='user_avg_session_duration'), on='user_id', how='left')

    return user_agg_df

# -----------------------------
# Kapsamlı product features
# -----------------------------
def calculate_comprehensive_product_features(df):
    product_agg_df = df.groupby('product_id').agg(
        product_total_events=('event_type', 'count'),
        product_unique_users=('user_id', 'nunique'),
        product_unique_sessions=('user_session', 'nunique'),
        product_unique_categories=('category_id', 'nunique'),
        product_min_event_time=('event_time', 'min'),
        product_max_event_time=('event_time', 'max'),
        product_avg_time_between_events=('event_time', lambda x: x.diff().mean().total_seconds()),
        product_std_time_between_events=('event_time', lambda x: x.diff().std().total_seconds())
    ).reset_index()

    product_agg_df['product_total_duration'] = (product_agg_df['product_max_event_time'] - product_agg_df['product_min_event_time']).dt.total_seconds()
    product_agg_df.drop(columns=['product_min_event_time', 'product_max_event_time'], inplace=True)

    product_event_counts = df.groupby(['product_id', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0).reset_index()
    product_event_counts.columns = ['product_id'] + [f"product_{col.lower()}_count" for col in all_event_types]
    product_agg_df = pd.merge(product_agg_df, product_event_counts, on='product_id', how='left')

    for etype in all_event_types:
        product_agg_df[f'product_{etype.lower()}_rate'] = product_agg_df[f'product_{etype.lower()}_count'] / product_agg_df['product_total_events']

    product_agg_df['product_view_to_add_cart_ratio'] = product_agg_df['product_view_count'] / product_agg_df['product_add_cart_count']
    product_agg_df['product_add_cart_to_buy_ratio'] = product_agg_df['product_add_cart_count'] / product_agg_df['product_buy_count']
    product_agg_df['product_buy_to_total_ratio'] = product_agg_df['product_buy_count'] / product_agg_df['product_total_events']
    product_agg_df['product_cart_abandon_ratio'] = (product_agg_df['product_add_cart_count'] - product_agg_df['product_buy_count']) / product_agg_df['product_add_cart_count']

    product_agg_df.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    return product_agg_df

# -----------------------------
# Kapsamlı category features
# -----------------------------
def calculate_comprehensive_category_features(df):
    category_agg_df = df.groupby('category_id').agg(
        category_total_events=('event_type', 'count'),
        category_unique_users=('user_id', 'nunique'),
        category_unique_sessions=('user_session', 'nunique'),
        category_unique_products=('product_id', 'nunique'),
        category_min_event_time=('event_time', 'min'),
        category_max_event_time=('event_time', 'max'),
        category_avg_time_between_events=('event_time', lambda x: x.diff().mean().total_seconds()),
        category_std_time_between_events=('event_time', lambda x: x.diff().std().total_seconds())
    ).reset_index()

    category_agg_df['category_total_duration'] = (category_agg_df['category_max_event_time'] - category_agg_df['category_min_event_time']).dt.total_seconds()
    category_agg_df.drop(columns=['category_min_event_time', 'category_max_event_time'], inplace=True)

    category_event_counts = df.groupby(['category_id', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0).reset_index()
    category_event_counts.columns = ['category_id'] + [f"category_{col.lower()}_count" for col in all_event_types]
    category_agg_df = pd.merge(category_agg_df, category_event_counts, on='category_id', how='left')

    for etype in all_event_types:
        category_agg_df[f'category_{etype.lower()}_rate'] = category_agg_df[f'category_{etype.lower()}_count'] / category_agg_df['category_total_events']

    category_agg_df['category_view_to_add_cart_ratio'] = category_agg_df['category_view_count'] / category_agg_df['category_add_cart_count']
    category_agg_df['category_add_cart_to_buy_ratio'] = category_agg_df['category_add_cart_count'] / category_agg_df['category_buy_count']
    category_agg_df['category_buy_to_total_ratio'] = category_agg_df['category_buy_count'] / category_agg_df['category_total_events']
    category_agg_df['category_cart_abandon_ratio'] = (category_agg_df['category_add_cart_count'] - category_agg_df['category_buy_count']) / category_agg_df['category_add_cart_count']

    category_agg_df.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    return category_agg_df

# -----------------------------
# Oturum içi oranlar
# -----------------------------
def add_session_event_features(df):
    """
    Oturum bazlı event sayıları, oranlar, dönüşümler ve zaman tabanlı istatistikleri hesaplar.
    """
    # Oturum-event pivot table
    session_event_counts = df.groupby(['user_session', 'event_type']).size().unstack(fill_value=0).reindex(columns=all_event_types, fill_value=0)

    # Temel istatistikler
    session_event_counts['total_events'] = session_event_counts.sum(axis=1)
    session_event_counts['mean_events'] = session_event_counts.mean(axis=1)
    session_event_counts['std_events'] = session_event_counts.std(axis=1)
    session_event_counts['max_events'] = session_event_counts.max(axis=1)
    session_event_counts['min_events'] = session_event_counts.min(axis=1)

    # Oranlar
    for etype in all_event_types:
        session_event_counts[f'{etype.lower()}_rate'] = session_event_counts[etype] / session_event_counts['total_events']

    # Özel dönüşüm oranları
    session_event_counts['view_to_add_cart_ratio'] = session_event_counts['VIEW'] / session_event_counts['ADD_CART']
    session_event_counts['view_to_remove_cart_ratio'] = session_event_counts['VIEW'] / session_event_counts['REMOVE_CART']
    session_event_counts['add_cart_to_buy_ratio'] = session_event_counts['ADD_CART'] / session_event_counts['BUY']
    session_event_counts['add_cart_to_remove_cart_ratio'] = session_event_counts['ADD_CART'] / session_event_counts['REMOVE_CART']

    # Null veya sonsuz değerleri sıfırla
    session_event_counts.replace([np.inf, -np.inf, np.nan], 0, inplace=True)

    # Zaman tabanlı özellikler
    session_times = df.groupby('user_session')['event_time']
    session_event_counts['session_duration_sec'] = (session_times.max() - session_times.min()).dt.total_seconds()
    session_event_counts['mean_inter_event_sec'] = session_times.apply(lambda x: x.diff().dt.total_seconds().mean()).fillna(0)
    session_event_counts['std_inter_event_sec'] = session_times.apply(lambda x: x.diff().dt.total_seconds().std()).fillna(0)

    # Gün ve saat bazlı dağılım
    session_event_counts['unique_days'] = session_times.apply(lambda x: x.dt.date.nunique())
    session_event_counts['morning_events'] = df[df['event_time'].dt.hour.between(6,11)].groupby('user_session')['event_time'].count()
    session_event_counts['afternoon_events'] = df[df['event_time'].dt.hour.between(12,17)].groupby('user_session')['event_time'].count()
    session_event_counts['evening_events'] = df[df['event_time'].dt.hour.between(18,23)].groupby('user_session')['event_time'].count()
    session_event_counts['night_events'] = df[df['event_time'].dt.hour.between(0,5)].groupby('user_session')['event_time'].count()
    session_event_counts.fillna(0, inplace=True)

    # Ürün ve kategori çeşitliliği
    session_event_counts['unique_products'] = df.groupby('user_session')['product_id'].nunique()
    session_event_counts['unique_categories'] = df.groupby('user_session')['category_id'].nunique()

    # Merge ile orijinal df'e ekle
    return pd.merge(df, session_event_counts.reset_index(), on='user_session', how='left')

# -----------------------------
# Train ve Test için özellik çıkarımı
# -----------------------------
print("Train verisi için özellik çıkarımı yapılıyor...")
train = calculate_session_features(train)
user_counts_train = calculate_total_counts(train, 'user_id', all_event_types)
category_counts_train = calculate_total_counts(train, 'category_id', all_event_types)
product_counts_train = calculate_total_counts(train, 'product_id', all_event_types)
user_comprehensive_train = calculate_comprehensive_user_features(train)
product_comprehensive_train = calculate_comprehensive_product_features(train)
category_comprehensive_train = calculate_comprehensive_category_features(train)

train = pd.merge(train, user_counts_train, on='user_id', how='left')
train = pd.merge(train, category_counts_train, on='category_id', how='left')
train = pd.merge(train, product_counts_train, on='product_id', how='left')
train = add_daily_and_session_to_daily_features(train)
train = pd.merge(train, user_comprehensive_train, on='user_id', how='left')
train = pd.merge(train, product_comprehensive_train, on='product_id', how='left')
train = pd.merge(train, category_comprehensive_train, on='category_id', how='left')
train = add_session_event_features(train)

print("\nTest verisi için özellik çıkarımı yapılıyor...")
test = calculate_session_features(test)
user_counts_test = calculate_total_counts(test, 'user_id', all_event_types)
category_counts_test = calculate_total_counts(test, 'category_id', all_event_types)
product_counts_test = calculate_total_counts(test, 'product_id', all_event_types)
user_comprehensive_test = calculate_comprehensive_user_features(test)
product_comprehensive_test = calculate_comprehensive_product_features(test)
category_comprehensive_test = calculate_comprehensive_category_features(test)

test = pd.merge(test, user_counts_test, on='user_id', how='left')
test = pd.merge(test, category_counts_test, on='category_id', how='left')
test = pd.merge(test, product_counts_test, on='product_id', how='left')
test = add_daily_and_session_to_daily_features(test)
test = pd.merge(test, user_comprehensive_test, on='user_id', how='left')
test = pd.merge(test, product_comprehensive_test, on='product_id', how='left')
test = pd.merge(test, category_comprehensive_test, on='category_id', how='left')
test = add_session_event_features(test)

print("Özellik çıkarımı tamamlandı.")


Zaman döngüsüne dayalı özellikler ekleniyor...
Zaman tabanlı mevcut özellikler başarıyla eklendi.
Train verisi için özellik çıkarımı yapılıyor...
Kapsamlı user-id bazlı özellikler hesaplanıyor...
Günlük ve oturum-günlük oran özellikleri hesaplanıyor...

Test verisi için özellik çıkarımı yapılıyor...
Kapsamlı user-id bazlı özellikler hesaplanıyor...
Günlük ve oturum-günlük oran özellikleri hesaplanıyor...
Özellik çıkarımı tamamlandı.


In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141219 entries, 0 to 141218
Columns: 151 entries, event_time to unique_categories
dtypes: datetime64[ns, UTC](3), float64(72), int32(3), int64(67), object(6)
memory usage: 161.1+ MB


In [14]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

# -----------------------------
# 1️⃣ Kategorik kolonları dönüştür
# -----------------------------
cat_cols = ["event_type", "product_id", "category_id", "user_id"]
for col in cat_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

# -----------------------------
# 2️⃣ Feature listesini oluştur
# -----------------------------
# event_time ve session_value dışındaki sayısal ve kategorik sütunları al
numeric_cols = train.select_dtypes(include=['int64', 'float64', 'int32','float32']).columns.tolist()
if 'event_time' in numeric_cols:
    numeric_cols.remove('event_time')
if 'session_value' in numeric_cols:
    numeric_cols.remove('session_value')
cols = numeric_cols + cat_cols

# -----------------------------
# 3️⃣ Time-based CV için veri hazırlığı
# -----------------------------
train["event_time"] = pd.to_datetime(train["event_time"])
train = train.sort_values("event_time").reset_index(drop=True)

# Veri aralıklarını belirle (varsayım: train 21 gün, test 9 gün)
min_date = train["event_time"].min().normalize()
max_date = train["event_time"].max().normalize()
total_days = (max_date - min_date).days + 1
print(f"Toplam veri gün sayısı: {total_days}")

# Fold ayarları
# 12 gün train, 3 gün validation olarak daha geniş pencerelerle deneyelim
train_days = 9
val_days = 3
folds = []
start_day = 0

while start_day + train_days + val_days <= total_days:
    train_start = min_date + pd.Timedelta(days=start_day)
    train_end = train_start + pd.Timedelta(days=train_days - 1)
    val_start = train_end + pd.Timedelta(days=1)
    val_end = val_start + pd.Timedelta(days=val_days - 1)
    
    folds.append((train_start, train_end, val_start, val_end))
    start_day += val_days

# -----------------------------
# 4️⃣ Fold bazlı eğitim ve ortalama MSE hesaplama
# -----------------------------
all_mse_scores = []

for i, (train_start, train_end, val_start, val_end) in enumerate(folds):
    X_train = train[(train["event_time"].dt.date >= train_start.date()) & (train["event_time"].dt.date <= train_end.date())][cols]
    y_train = train[(train["event_time"].dt.date >= train_start.date()) & (train["event_time"].dt.date <= train_end.date())]["session_value"]
    
    X_val = train[(train["event_time"].dt.date >= val_start.date()) & (train["event_time"].dt.date <= val_end.date())][cols]
    y_val = train[(train["event_time"].dt.date >= val_start.date()) & (train["event_time"].dt.date <= val_end.date())]["session_value"]
    
    # Eğer validasyon setinde hiç veri yoksa, bu foldu atla
    if X_val.empty:
        print(f"Fold {i+1}: Validasyon seti boş. Atlanıyor.")
        continue

    model = lgb.LGBMRegressor(
        objective="regression",
        boosting_type="gbdt",
        learning_rate=0.1,
        n_estimators=100,
        num_leaves=31,
        max_depth=5,
        random_state=42,
        verbose=-1
    )
    
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    
    y_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_pred, squared=True)
    all_mse_scores.append(mse)
    
    print(f"Fold {i+1}: Train {train_start.date()} → {train_end.date()}, "
          f"Val {val_start.date()} → {val_end.date()}, Validation MSE: {mse:.4f}")

# -----------------------------
# 5️⃣ Sonuçları Yazdır
# -----------------------------
if all_mse_scores:
    avg_mse = np.mean(all_mse_scores)
    std_mse = np.std(all_mse_scores)
    print("\n" + "="*50)
    print(f"Tüm Foldların Ortalama MSE'si: {avg_mse:.4f} ± {std_mse:.4f}")
    print("="*50)
else:
    print("Foldlar oluşturulamadı veya validasyon setleri boştu.")

Toplam veri gün sayısı: 21


c:\Users\tursu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Fold 1: Train 2025-06-01 → 2025-06-09, Val 2025-06-10 → 2025-06-12, Validation MSE: 1392.7287


c:\Users\tursu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Fold 2: Train 2025-06-04 → 2025-06-12, Val 2025-06-13 → 2025-06-15, Validation MSE: 15382.0023


c:\Users\tursu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Fold 3: Train 2025-06-07 → 2025-06-15, Val 2025-06-16 → 2025-06-18, Validation MSE: 1607.9584
Fold 4: Train 2025-06-10 → 2025-06-18, Val 2025-06-19 → 2025-06-21, Validation MSE: 826.2399

Tüm Foldların Ortalama MSE'si: 4802.2323 ± 6114.9029


c:\Users\tursu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [15]:
# Ensure outputs saved under outputs/
import os
submission_path = os.path.join(OUTPUT_DIR, "lgbm_model_submission.csv")
if 'submission' in globals():
    try:
        submission.to_csv(submission_path, index=False)
        print(f"Submission saved to: {submission_path}")
    except Exception as e:
        print(f"Submission save failed: {e}")


Submission saved to: outputs\lgbm_model_submission.csv


In [16]:
# -----------------------------
# 6️⃣ Feature Importance Çıkarımı
# -----------------------------
print("6️⃣ Özellik Önem Dereceleri (Feature Importance) hesaplanıyor...")

# Modelin özellik önem değerlerini al
feature_importance = model.feature_importances_

# Özellik adlarını al
feature_names = X_train.columns

# DataFrame oluştur
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values(by='importance', ascending=False)

# En önemli 30 özellik
top_30 = importance_df.head(30)

# En önemsiz 30 özellik
bottom_30 = importance_df.tail(30)

print("\n✅ En Önemli 30 Özellik:")
print(top_30)

print("\n❌ En Önemsiz 30 Özellik:")
print(bottom_30)

# Tahmin ve MSE hesaplama kodu burada devam ediyor
print("\nModel eğitimi tamamlandı!")
y_pred = model.predict(X_val)
mse = mean_squared_error(y_val, y_pred, squared=True)
print(f"Validation MSE: {mse:.4f}")


6️⃣ Özellik Önem Dereceleri (Feature Importance) hesaplanıyor...

✅ En Önemli 30 Özellik:
                             feature  importance
144                          user_id         292
3                session_event_count         124
8                   session_duration         113
1                               hour         107
126                         buy_rate          97
2           time_since_session_start          92
47                             BUY_x          80
20             std_inter_event_sec_x          72
24                  user_buy_count_x          71
143                      category_id          63
70                     user_buy_rate          58
51        session_to_daily_buy_ratio          58
123                    add_cart_rate          58
5             session_category_count          52
142                       product_id          51
120                       std_events          48
125                 remove_cart_rate          46
0                        day

c:\Users\tursu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [17]:
# Tüm train verisiyle final eğitim
X_full = train[cols]
y_full = train["session_value"]

final_model = lgb.LGBMRegressor(
    objective="regression",
    boosting_type="gbdt",
    learning_rate=0.1,
    n_estimators=100,
    num_leaves=31,
    max_depth=5,
    random_state=42,
    verbose=-1
)

final_model.fit(X_full, y_full, categorical_feature=cat_cols)


print("7️⃣ Test seti tahmin ediliyor...")
# -----------------------------
# 6️⃣ Test Tahmini (row bazlı)
# -----------------------------
test["pred"] = final_model.predict(test[cols])

# -----------------------------
# 1️⃣ Ortak user_session değerlerini bulma ve ayırma
# -----------------------------
# train'den her user_session için tekil session_value'yu al
train_session_values = train.groupby("user_session")["session_value"].first().reset_index()

# Ortak user_session'ları belirle
common_sessions = pd.merge(train_session_values, test, on="user_session", how="inner")

# Ortak session'ları test setinden ayır
test_common_sessions = test[test["user_session"].isin(common_sessions["user_session"])]
test_remaining_sessions = test[~test["user_session"].isin(common_sessions["user_session"])]


# -----------------------------
# 2️⃣ Kalan veriyi modelle tahmin etme
# -----------------------------
# Sadece kalan test verisi için model tahmini yap
if not test_remaining_sessions.empty:
    test_remaining_sessions["pred"] = final_model.predict(test_remaining_sessions[cols])
else:
    print("Test setinde train'de olmayan user_session bulunmuyor.")
    test_remaining_sessions["pred"] = np.nan # Veya boş bırakın

# -----------------------------
# 3️⃣ Tahminleri ve gerçek değerleri birleştirme
# -----------------------------
# Ortak user_session'ların session_value'larını train'den al
test_common_sessions_with_value = pd.merge(test_common_sessions, train_session_values, on="user_session", how="left")

# Tahminleri ve gerçek değerleri birleştir
final_test_preds = pd.concat([test_remaining_sessions, test_common_sessions_with_value], ignore_index=True)
final_test_preds["pred"] = final_test_preds["pred"].fillna(final_test_preds["session_value"])


# -----------------------------
# 4️⃣ Session bazlı aggregation ve Submission
# -----------------------------
session_preds = final_test_preds.groupby("user_session")["pred"].mean().reset_index()
session_preds.columns = ["user_session", "session_value"]

# Submission dosyası
submission = submission.drop(columns="session_value", errors="ignore").merge(session_preds, on="user_session", how="left")
submission.to_csv("lgbm_model_submission.csv", index=False)

print("Submission hazır! İlk 5 satır:")
print(submission.head())


7️⃣ Test seti tahmin ediliyor...


C:\Users\tursu\AppData\Local\Temp\ipykernel_11104\987714691.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_remaining_sessions["pred"] = final_model.predict(test_remaining_sessions[cols])


Submission hazır! İlk 5 satır:
     user_session  session_value
0  SESSION_164059     184.356474
1  SESSION_109583      44.911368
2  SESSION_171382      38.596586
3  SESSION_137110      29.444587
4  SESSION_146503     193.983122
